In [13]:
# ============================================================
# COMPLETE REGRESSION MACHINE LEARNING PIPELINE
# Encoders + Scaling + Multiple Algorithms
# ============================================================
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)
import pandas as pd
import numpy as np


# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    LabelEncoder,
    OneHotEncoder,
    StandardScaler,
    MinMaxScaler
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


# Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

from sklearn.neural_network import MLPRegressor


# Evaluation
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)



# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_csv("houses_improved.csv")

print("Dataset Preview:")
print(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns)



# ============================================================
# 2. DEFINE TARGET COLUMN
# ============================================================

target = "Price_ETB"


X = df.drop(target, axis=1)

y = df[target]



# ============================================================
# 3. HANDLE MISSING VALUES
# ============================================================

numeric_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns


categorical_columns = X.select_dtypes(include=['object', 'string']).columns



for col in numeric_columns:
    X[col] = X[col].fillna(
        X[col].mean()
    )


for col in categorical_columns:
    X[col] = X[col].fillna(
        X[col].mode()[0]
    )



# ============================================================
# 4. LABEL ENCODING
# ============================================================

X_label = X.copy()


label_encoder = LabelEncoder()


for col in categorical_columns:

    X_label[col] = label_encoder.fit_transform(
        X_label[col]
    )


print("\nLabel Encoding Completed")



# ============================================================
# 5. ONE HOT ENCODING
# ============================================================


ohe = ColumnTransformer(

    transformers=[

        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_columns
        )

    ],

    remainder="passthrough"

)


print("OHE Prepared")



# ============================================================
# 6. TRAIN TEST SPLIT
# ============================================================


X_train_label, X_test_label, y_train, y_test = train_test_split(
    X_label,
    y,
    test_size=0.2,
    random_state=42
)



X_train_ohe, X_test_ohe, _, _ = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)



# ============================================================
# 7. SCALING METHODS
# ============================================================


scalers = {

    "No Scaling": None,

    "StandardScaler": StandardScaler(),

    "MinMaxScaler": MinMaxScaler()

}



# ============================================================
# 8. MODELS
# ============================================================


models = {


    "Linear Regression":
        LinearRegression(),


    "Ridge Regression":
        Ridge(),


    "Lasso Regression":
        Lasso(),


    "Support Vector Regression":
        SVR(),


    "Random Forest":
        RandomForestRegressor(
            n_estimators=100,
            random_state=42
        ),


    "Gradient Boosting":
        GradientBoostingRegressor(
            random_state=42
        ),


    "MLP Neural Network":
        MLPRegressor(
            hidden_layer_sizes=(100,50),
            max_iter=500,
            random_state=42
        )

}



# ============================================================
# 9. TRAINING AND EVALUATION
# ============================================================


results = []



# ------------------------------------------------------------
# LABEL ENCODING MODELS
# ------------------------------------------------------------


for scaler_name, scaler in scalers.items():

    for model_name, model in models.items():


        print(
            "Training:",
            "Label Encoding |",
            scaler_name,
            "|",
            model_name
        )


        if scaler_name == "No Scaling":

            pipeline = Pipeline(
                steps=[
                    ("model", model)
                ]
            )

        else:

            pipeline = Pipeline(
                steps=[
                    ("scaler", scaler),
                    ("model", model)
                ]
            )



        pipeline.fit(
            X_train_label,
            y_train
        )


        prediction = pipeline.predict(
            X_test_label
        )



        mae = mean_absolute_error(
            y_test,
            prediction
        )


        mse = mean_squared_error(
            y_test,
            prediction
        )


        rmse = np.sqrt(mse)


        r2 = r2_score(
            y_test,
            prediction
        )



        results.append(
            [
                "Label Encoding",
                scaler_name,
                model_name,
                mae,
                mse,
                rmse,
                r2
            ]
        )



# ------------------------------------------------------------
# OHE MODELS
# ------------------------------------------------------------


for scaler_name, scaler in scalers.items():

    for model_name, model in models.items():


        print(
            "Training:",
            "OHE |",
            scaler_name,
            "|",
            model_name
        )


        if scaler_name == "No Scaling":

            pipeline = Pipeline(
                steps=[

                    ("encoder", ohe),

                    ("model", model)

                ]
            )


        else:

            pipeline = Pipeline(
                steps=[

                    ("encoder", ohe),

                    ("scaler", scaler),

                    ("model", model)

                ]
            )



        pipeline.fit(
            X_train_ohe,
            y_train
        )


        prediction = pipeline.predict(
            X_test_ohe
        )



        mae = mean_absolute_error(
            y_test,
            prediction
        )


        mse = mean_squared_error(
            y_test,
            prediction
        )


        rmse = np.sqrt(mse)


        r2 = r2_score(
            y_test,
            prediction
        )



        results.append(
            [
                "One Hot Encoding",
                scaler_name,
                model_name,
                mae,
                mse,
                rmse,
                r2
            ]
        )



# ============================================================
# 10. DISPLAY RESULTS
# ============================================================


results_df = pd.DataFrame(

    results,

    columns=[
        "Encoder",
        "Scaling",
        "Model",
        "MAE",
        "MSE",
        "RMSE",
        "R2 Score"
    ]

)



results_df = results_df.sort_values(
    by="R2 Score",
    ascending=False
)



print("\n==============================")
print("FINAL MODEL RESULTS")
print("==============================")

print(results_df)



# ============================================================
# 11. BEST MODEL
# ============================================================


print("\n==============================")
print("BEST MODEL")
print("==============================")


print(results_df.iloc[0])

Dataset Preview:
   Number_of_Rooms  Site_Area_sqm  Built_Area_sqm  Property_Years  \
0                3            402             275              18   
1                6            111              72               5   
2                4            417             306               2   
3                4            179              72               4   
4                3            315             147              12   

  Construction_Materials Housing_Typology Land_Value_Grading  \
0               Concrete    Semi-detached                Low   
1               Concrete    Semi-detached               High   
2               Concrete      Condominium             Medium   
3               Mud&Wood      Condominium                Low   
4               Concrete    Semi-detached             Medium   

   Proximity_to_CBD_km  Proximity_to_Bus_Station_km Type_of_Nearest_Road  \
0                 0.66                         0.39               Gravel   
1                 0.12         

In [16]:
import warnings
from sklearn.exceptions import InconsistentVersionWarning

warnings.filterwarnings("ignore", category=InconsistentVersionWarning)
import joblib

model = joblib.load("house_price_model.pkl")